In [1]:
#1-D3
# Demonstrate the basic convolution operation used in CNNs.

import torch
import torch.nn as nn


# Create a simple single-channel input image.
input_image = torch.tensor([
    [
        [1.0, 2.0, 3.0, 4.0],
        [2.0, 3.0, 4.0, 5.0],
        [3.0, 4.0, 5.0, 6.0],
        [4.0, 5.0, 6.0, 7.0]
    ]
])


# Add the batch dimension required by Conv2d.
input_image = input_image.unsqueeze(0)


# Define a simple 3 × 3 convolution filter.
convolution = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3,
    bias=False
)


# Run the convolution.
feature_map = convolution(input_image)


print("Input shape:", input_image.shape)
print("Feature-map shape:", feature_map.shape)
print("\nFeature map:")
print(feature_map)


Input shape: torch.Size([1, 1, 4, 4])
Feature-map shape: torch.Size([1, 1, 2, 2])

Feature map:
tensor([[[[0.9628, 0.9929],
          [0.9929, 1.0230]]]], grad_fn=<ConvolutionBackward0>)


In [2]:
# 2-D3
# Demonstrate multiple convolution filters producing multiple feature maps.

import torch
import torch.nn as nn


# Create a simple RGB input image.
input_image = torch.randn(
    1,
    3,
    8,
    8
)


# Create a convolution layer with four filters.
convolution = nn.Conv2d(
    in_channels=3,
    out_channels=4,
    kernel_size=3,
    padding=1
)


# Apply the convolution.
feature_maps = convolution(input_image)


print("Input shape:", input_image.shape)
print("Number of filters:", convolution.out_channels)
print("Feature-map shape:", feature_maps.shape)

Input shape: torch.Size([1, 3, 8, 8])
Number of filters: 4
Feature-map shape: torch.Size([1, 4, 8, 8])


In [3]:
# 3-D3
# Define and inspect a minimal convolutional neural network.

import torch
import torch.nn as nn


class SimpleCNN(nn.Module):

    def __init__(self):
        super().__init__()

        # First convolution:
        # 1 input channel because the example image is grayscale.
        # 4 output channels means 4 learnable filters.
        # A 3x3 kernel examines small regions of the image.
        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=4,
            kernel_size=3,
            padding=1
        )

        # Second convolution:
        # It receives the 4 feature maps from conv1
        # and produces 8 new feature maps.
        self.conv2 = nn.Conv2d(
            in_channels=4,
            out_channels=8,
            kernel_size=3,
            padding=1
        )

        # 2x2 max pooling reduces the height and width by half.
        self.pool = nn.MaxPool2d(2)

        # After two pooling operations:
        #
        # 28 x 28
        #   -> 14 x 14
        #   -> 7 x 7
        #
        # The second convolution produces 8 feature maps.
        # Therefore the flattened size is:
        #
        # 8 x 7 x 7 = 392
        self.fc = nn.Linear(
            8 * 7 * 7,
            10
        )

    def forward(self, x):

        # First feature-extraction stage.
        x = self.conv1(x)
        x = torch.relu(x)
        x = self.pool(x)

        # Second feature-extraction stage.
        x = self.conv2(x)
        x = torch.relu(x)
        x = self.pool(x)

        # Convert the feature maps into a one-dimensional vector.
        x = torch.flatten(x, 1)

        # Produce the final output.
        x = self.fc(x)

        return x


# Create the CNN.
model = SimpleCNN()


print(model)

SimpleCNN(
  (conv1): Conv2d(1, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc): Linear(in_features=392, out_features=10, bias=True)
)


In [4]:
# 4-D3
# Trace the tensor shape through each stage of the CNN.

# Create one grayscale 28 x 28 input image.
input_image = torch.randn(
    1,
    1,
    28,
    28
)

print("Input shape:              ", input_image.shape)


# First convolution.
x = model.conv1(input_image)

print("After conv1:              ", x.shape)


# First ReLU activation.
x = torch.relu(x)

print("After ReLU 1:             ", x.shape)


# First max-pooling operation.
x = model.pool(x)

print("After pool 1:             ", x.shape)


# Second convolution.
x = model.conv2(x)

print("After conv2:              ", x.shape)


# Second ReLU activation.
x = torch.relu(x)

print("After ReLU 2:             ", x.shape)


# Second max-pooling operation.
x = model.pool(x)

print("After pool 2:             ", x.shape)


# Flatten the feature maps.
x = torch.flatten(x, 1)

print("After flatten:            ", x.shape)


# Final fully connected layer.
x = model.fc(x)

print("Final output:             ", x.shape)

Input shape:               torch.Size([1, 1, 28, 28])
After conv1:               torch.Size([1, 4, 28, 28])
After ReLU 1:              torch.Size([1, 4, 28, 28])
After pool 1:              torch.Size([1, 4, 14, 14])
After conv2:               torch.Size([1, 8, 14, 14])
After ReLU 2:              torch.Size([1, 8, 14, 14])
After pool 2:              torch.Size([1, 8, 7, 7])
After flatten:             torch.Size([1, 392])
Final output:              torch.Size([1, 10])


In [38]:
# 5-D3
# Run one input image through the complete CNN
# and inspect the model output.


# Create one random grayscale image.
input_image = torch.randn(
    1,
    1,
    28,
    28
)


# Run the complete image through the CNN.
output = model(input_image)


print("Output shape:", output.shape)
print("\nRaw model output:")
print(output)


# Convert the raw outputs into probabilities.
probabilities = torch.softmax(
    output,
    dim=1
)


print("\nProbabilities:")
print(probabilities)


# Find the output with the highest probability.
predicted_class = torch.argmax(
    probabilities,
    dim=1
)


print("\nPredicted class:", predicted_class.item())

Output shape: torch.Size([1, 10])

Raw model output:
tensor([[ 0.2547,  0.2832,  0.3292, -0.1168,  0.0971,  0.7189,  0.1167, -0.3964,
         -0.1691,  0.0879]], grad_fn=<AddmmBackward0>)

Probabilities:
tensor([[0.1095, 0.1126, 0.1179, 0.0755, 0.0935, 0.1741, 0.0954, 0.0571, 0.0717,
         0.0927]], grad_fn=<SoftmaxBackward0>)

Predicted class: 5


In [39]:
# 6-D3
# Demonstrate how a CNN prediction is compared
# with a known target label using a loss function.

import torch
import torch.nn as nn


# Create one example input image.
input_image = torch.randn(
    1,
    1,
    28,
    28
)


# Define the correct class for this example.
# Class indices range from 0 to 9.
target = torch.tensor([3])


# Run the image through the CNN.
output = model(input_image)


# Create the classification loss function.
loss_function = nn.CrossEntropyLoss()


# Calculate how different the model output
# is from the correct target class.
loss = loss_function(
    output,
    target
)


print("Model output shape:", output.shape)
print("Target class:", target.item())
print("Loss:", loss.item())

Model output shape: torch.Size([1, 10])
Target class: 3
Loss: 2.4470953941345215


In [40]:
# 7-D3
# Calculate gradients of the loss with respect to
# the CNN parameters.

# Clear any existing gradients.
model.zero_grad()


# Calculate the model output again.
output = model(input_image)


# Calculate the loss again.
loss = loss_function(
    output,
    target
)


# Backpropagate the loss through the network.
loss.backward()


# Inspect the gradient of the first convolution layer.
print("Conv1 weight shape:", model.conv1.weight.shape)
print("Conv1 gradient shape:", model.conv1.weight.grad.shape)

print("\nFirst convolution gradient:")
print(model.conv1.weight.grad)

print("\nGradient mean:",
      model.conv1.weight.grad.mean().item())

print("Gradient absolute maximum:",
      model.conv1.weight.grad.abs().max().item())

Conv1 weight shape: torch.Size([4, 1, 3, 3])
Conv1 gradient shape: torch.Size([4, 1, 3, 3])

First convolution gradient:
tensor([[[[-0.1515, -0.1035, -0.0199],
          [ 0.0767,  0.2668, -0.2087],
          [ 0.1685,  0.0589,  0.0889]]],


        [[[ 0.1759,  0.1202, -0.1909],
          [ 0.0699, -0.1962, -0.1553],
          [ 0.1320,  0.0023, -0.0091]]],


        [[[ 0.1942,  0.0247,  0.0905],
          [ 0.1156, -0.0588,  0.0923],
          [ 0.1337,  0.0491,  0.2006]]],


        [[[ 0.0802,  0.1511,  0.1487],
          [ 0.0457,  0.0343, -0.1079],
          [ 0.1936, -0.0376, -0.1086]]]])

Gradient mean: 0.03794831782579422
Gradient absolute maximum: 0.26677268743515015


In [41]:
# 8-D3
# Demonstrate how an optimizer updates CNN weights
# using the gradients calculated by backpropagation.


# Create a simple optimizer.
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)


# Store one weight before the update.
weight_before = model.conv1.weight[0, 0, 0, 0].item()


# Clear old gradients.
model.zero_grad()


# Run the input through the model.
output = model(input_image)


# Calculate the loss.
loss = loss_function(
    output,
    target
)


# Calculate gradients.
loss.backward()


# Store the gradient of the selected weight.
gradient = model.conv1.weight.grad[0, 0, 0, 0].item()


# Update all model weights using the gradients.
optimizer.step()


# Read the same weight after the update.
weight_after = model.conv1.weight[0, 0, 0, 0].item()


print("Weight before update:", weight_before)
print("Gradient:", gradient)
print("Weight after update:", weight_after)

Weight before update: 0.1887754499912262
Gradient: -0.1515044867992401
Weight after update: 0.19029049575328827
